# 06 — SVD Dimensionality Reduction and GMM Clustering
Applies truncated Singular Value Decomposition (SVD) to reduce the molecular embedding
to a low-dimensional latent space, then fits a Gaussian Mixture Model (GMM)
to cluster the molecules and identify those co-localised with the reference compound.

The optimal truncation rank is selected automatically as the smallest integer such that
the relative reconstruction error falls below a predefined threshold.
The optimal GMM is selected by iterating over a range of random states and covariance
types, choosing the configuration minimising the BIC criterion.

Run this notebook once per embedding type (fingerprint, image, graph)
by changing `EMBEDDING_CSV`, `REFERENCE_CSV`, and `OUTPUT_DIR`.

**Output per run:**
- `labels_SVD.npy` — cluster assignment for each molecule
- `indices_cluster_SVD.npy` — indices of molecules in the reference cluster

## Configuration
Change `EMBEDDING_CSV`, `REFERENCE_CSV`, and `OUTPUT_DIR` to switch between
fingerprint, image, and graph representations.

In [ ]:
# Input 
EMBEDDING_CSV = "data/fingerprint_embeddings.csv"  # dataset embedding CSV
REFERENCE_CSV = "data/adagrasib_fingerprint.csv"   # reference molecule embedding CSV

# Output 
OUTPUT_DIR    = "data/svd_results/fingerprint/"    # output directory for this run

# SVD rank selection 
ERROR_THRESHOLD = 0.01  # relative error: select k s.t. sigma_{k+1}/sigma_1 < threshold

# GMM parameters 
GMM_MAX_COMPONENTS    = 20
GMM_CRITERION         = 'bic'
GMM_COV_TYPES         = ['full', 'tied', 'diag', 'spherical']
GMM_RANDOM_STATE_RANGE = (0, 50)

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Dependencies

In [ ]:
# pip install numpy pandas scikit-learn matplotlib seaborn

## 1. Load embeddings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture

df  = pd.read_csv(EMBEDDING_CSV, dtype=float)
ada = pd.read_csv(REFERENCE_CSV, dtype=float)

print(f"Dataset shape   : {df.shape}")
print(f"Reference shape : {ada.shape}")

## 2. SVD decomposition and sign correction
Sign correction on the first singular vector ensures deterministic orientation
of the dominant component across different platforms and implementations.

In [ ]:
M = df.values
U, S, VT = np.linalg.svd(M, full_matrices=False)

# Sign correction on first component (deterministic orientation)
if U[:, 0].sum() < 0:
    U[:, 0]  = -U[:, 0]
    VT[0, :] = -VT[0, :]

# Explained variance
explained_var       = (S**2) / (df.shape[0] - 1)
explained_var_ratio = explained_var / np.sum(explained_var)
cumulative_var      = np.cumsum(explained_var_ratio)

print(f"Top-5 explained variance ratios: {explained_var_ratio[:5].round(4)}")
print(f"Cumulative variance (top-5)    : {cumulative_var[:5].round(4)}")

## 3. Automatic rank selection
Selects the smallest rank `k` such that the relative reconstruction error
falls below `ERROR_THRESHOLD`:

$$\epsilon(k) = \frac{\sigma_{k+1}}{\sigma_1} < \text{threshold}$$

In [ ]:
def select_rank(S, error_threshold=ERROR_THRESHOLD):
    k = 1
    while k < len(S):
        rel_error = S[k] / S[0]
        print(f"k={k:3d} | sigma_{k}={S[k]:.6f} | rel_error = {rel_error:.6f}")
        if rel_error < error_threshold:
            print(f"\n Optimal rank found: k={k} (rel_error={rel_error:.6f})")
            return k
        k += 1
    raise ValueError(f"rel_error never dropped below {error_threshold} within available singular values.")


optimal_k = select_rank(S)
print(f"\nSelected rank: {optimal_k}")

## 4. Compute reduced representation

In [ ]:
X_reduced = U[:, :optimal_k] @ np.diag(S[:optimal_k])  # (N, k)
print(f"Reduced dataset shape: {X_reduced.shape}")

## 5. Project reference molecule into SVD space

In [ ]:
ada_aligned = ada.iloc[:, :df.shape[1]].values
V_k = VT[:optimal_k, :]      # (k, D)
z   = ada_aligned @ V_k.T    # (1, k)

print(f"Reference projection shape: {z.shape}")
print(z)

## 6. GMM clustering with iterative random state
For each combination of covariance type and random state, GMMs are fitted
for all numbers of components up to `GMM_MAX_COMPONENTS`.
The configuration minimising the BIC is selected as the optimal model.

In [ ]:
def find_best_gmm(X, max_components=GMM_MAX_COMPONENTS, criterion=GMM_CRITERION,
                  random_state_range=GMM_RANDOM_STATE_RANGE,
                  covariance_types=GMM_COV_TYPES):
    best_score        = np.inf
    best_model        = None
    best_k            = None
    best_random_state = None
    best_cov_type     = None
    results           = []

    random_states = range(random_state_range[0], random_state_range[1] + 1)

    for cov_type in covariance_types:
        for rs in random_states:
            for k in range(1, max_components + 1):
                gmm = GaussianMixture(n_components=k, covariance_type=cov_type,
                                      random_state=rs)
                gmm.fit(X)
                score = gmm.bic(X) if criterion == 'bic' else gmm.aic(X)
                results.append({'n_components': k, 'random_state': rs,
                                'covariance_type': cov_type, criterion.upper(): score})
                if score < best_score:
                    best_score = score
                    best_model = gmm
                    best_k     = k
                    best_random_state = rs
                    best_cov_type     = cov_type

    results_df = pd.DataFrame(results)

    # Plot mean BIC across random states
    plt.figure(figsize=(12, 6))
    for cov_type in covariance_types:
        subset = results_df[results_df['covariance_type'] == cov_type]
        mean_scores = subset.groupby('n_components')[criterion.upper()].mean()
        plt.plot(mean_scores.index, mean_scores.values, marker='o',
                 label=f"covariance_type='{cov_type}'")
    plt.axvline(best_k, color='red', linestyle='--', label=f'Optimal k={best_k}')
    plt.title(f"{criterion.upper()} (mean over random states "
              f"{random_state_range[0]}–{random_state_range[1]})")
    plt.xlabel("Number of GMM components")
    plt.ylabel(criterion.upper())
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return best_model, best_k, best_score, best_random_state, best_cov_type, results_df


best_gmm, best_k, best_score, best_rs, best_cov, results_df = find_best_gmm(X_reduced)

print(f"Optimal components  : {best_k}")
print(f"Optimal {GMM_CRITERION.upper()}         : {best_score:.2f}")
print(f"Optimal random state: {best_rs}")
print(f"Optimal cov type    : {best_cov}")

## 7. Assign clusters and identify reference cluster

In [ ]:
labels    = best_gmm.predict(X_reduced)
probs     = best_gmm.predict_proba(X_reduced)

ref_label = best_gmm.predict(z)[0]
ref_probs = best_gmm.predict_proba(z)[0]

indices_cluster_SVD = np.where(labels == ref_label)[0]

print(f"Reference assigned to cluster : {ref_label}")
print(f"Membership probability        : {ref_probs[ref_label]:.4f}")
print(f"Molecules in reference cluster: {len(indices_cluster_SVD)}")

## 8. Visualisation
2D scatter plot of the SVD-reduced space.

In [ ]:
X_plot = X_reduced if X_reduced.shape[1] >= 2 else np.hstack([X_reduced, X_reduced])
z_plot = z

plt.figure(figsize=(22, 18), dpi=150)
palette = sns.color_palette('Blues', n_colors=len(np.unique(labels)))

sns.scatterplot(x=X_plot[:, 0], y=X_plot[:, 1], hue=labels,
                palette=palette, s=320, alpha=0.92,
                edgecolor='dimgray', linewidth=1.2)

same_cluster = labels == ref_label
plt.scatter(X_plot[same_cluster, 0], X_plot[same_cluster, 1],
            s=520, facecolors='none', edgecolors='lightcoral',
            linewidths=4, label=f'Reference cluster ({ref_label})', zorder=5)

plt.scatter(z_plot[0, 0], z_plot[0, 1],
            s=1600, facecolors='none', edgecolors='firebrick',
            linewidths=7, label='Reference molecule', zorder=10)

plt.xticks(fontsize=18); plt.yticks(fontsize=18)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left',
           fontsize=16, title='Clusters', title_fontsize=18, frameon=True)
plt.grid(alpha=0.18)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'SVD_clusters.tif'),
            dpi=300, bbox_inches='tight', pil_kwargs={'compression': 'tiff_lzw'})
plt.show()

## 9. Save outputs

In [ ]:
np.save(os.path.join(OUTPUT_DIR, 'labels_SVD.npy'),           labels)
np.save(os.path.join(OUTPUT_DIR, 'indices_cluster_SVD.npy'),  indices_cluster_SVD)